# ZelusBench: Sustained Attention

**Does accuracy degrade as dependency chains grow longer?**

Varies chain depth while randomizing ALL background conditions
(distractors, transforms, dimensionality, point types).
Uses LINEAR structure to guarantee exact depth targeting.
Every query targets exactly the depth being tested.

- Depths: 2, 4, 8, 16, 32
- 10 seeds per depth = 50 scenarios, ~150 queries

In [ ]:
import kaggle_benchmarks as kbench
import numpy as np
import random
import re
import time

from zelusbench.scenarios.config import ScenarioConfig
from zelusbench.scenarios.generator import ScenarioGenerator
from zelusbench.evaluation.parser import parse_model_response
from zelusbench.evaluation.scorer import score_query


def score_response(response_text, scenario):
    """Parse model response and score each query against ground truth."""
    query_dicts = [q.to_dict() for q in scenario.queries]
    parts = re.split(r'\[Query\s+q_\d+\]', response_text)
    if len(parts) > 1:
        parts = parts[1:]
    if len(parts) != len(query_dicts):
        parts = [response_text] * len(query_dicts)
    return [score_query(qd, parse_model_response(rp, qd))
            for qd, rp in zip(query_dicts, parts)]


CHAIN_DEPTHS = [2, 4, 8, 16, 32]
SEEDS = 10
TOTAL = len(CHAIN_DEPTHS) * SEEDS

print(f"ZelusBench Sustained Attention")
print(f"Depths: {CHAIN_DEPTHS} | Seeds: {SEEDS} | Total: {TOTAL} scenarios")

In [ ]:
@kbench.task(name="zelusbench_sustained_attention")
def zelusbench_sustained_attention(llm) -> tuple[float, float]:
    """Sustained attention: accuracy vs chain depth (randomized background).
    Returns (mean_accuracy, std_dev) across all depth levels."""

    _start = time.time()
    print(f"Running {TOTAL} scenarios across {len(CHAIN_DEPTHS)} chain depths...")
    print("=" * 60)

    all_scores = []
    depth_scores = {}
    scenario_num = 0

    for depth in CHAIN_DEPTHS:
        depth_scores[depth] = []
        for seed in range(SEEDS):
            scenario_num += 1
            unique_seed = depth * 1000 + seed
            rng = random.Random(unique_seed)
            from zelusbench.scenarios.config import DAGStructure
            cfg = ScenarioConfig.randomize_except(rng, pinned={
                "min_chain_depth": depth,
                "max_chain_depth": depth,
                "dag_structure": DAGStructure.LINEAR,
                "query_target_depth": depth,
                "num_queries": 3,
                "seed": unique_seed,
            })
            gen = ScenarioGenerator(cfg)
            scenario = gen.generate(f"sustained_d{depth}_s{seed}")

            response = llm.prompt(scenario.prompt)
            scores = score_response(response, scenario)
            all_scores.extend(scores)

            avg = float(np.mean([s.score for s in scores]))
            depth_scores[depth].append(avg)

            q_depths = [s.chain_depth for s in scores]
            tiers = [s.tier.name for s in scores]
            bg = f"dim={cfg.dim} dist={cfg.distractor_level.name} tx={cfg.transform_density.name}"
            print(f"  [{scenario_num}/{TOTAL}] depth={depth} seed={seed} "
                  f"avg={avg:.2f} q_depths={q_depths} tiers={tiers} | {bg}")

        depth_avg = float(np.mean(depth_scores[depth]))
        print(f"  >> Depth {depth} mean: {depth_avg:.3f}")

    print("\n" + "=" * 60)
    print("RESULTS BY DEPTH:")
    for depth in CHAIN_DEPTHS:
        avg = float(np.mean(depth_scores[depth]))
        print(f"  depth={depth:2d}  accuracy={avg:.3f}")
        kbench.assertions.assert_true(
            avg >= 0,
            expectation=f"Depth {depth}: model should produce valid answers (got {avg:.3f})"
        )

    overall = float(np.mean([s.score for s in all_scores]))
    std = float(np.std([s.score for s in all_scores]))
    elapsed = time.time() - _start

    print(f"\nOverall: {overall:.3f} +/- {std:.3f}")
    print(f"Total queries: {len(all_scores)}")
    print(f"Time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

    return overall, std


zelusbench_sustained_attention.run(llm=kbench.llm)

In [ ]:
%choose zelusbench_sustained_attention